# Guided applied case study: replace this title with the investigation

> **How to use this notebook.** Duplicate this file, give the copy a numbered and descriptive name, and replace every **STUDENT WORK** prompt. The example settings keep the untouched template executable; they are not a submission. Your finished notebook must stand on its own and use data analysis and machine learning to answer one real-world or scientific question on your chosen data. One or several algorithms may support that story.

**STUDENT WORK — governing question.** State the scientific or operational question in one sentence. Name the intended reader or user and explain what a useful answer could change. Do not begin with an algorithm name; begin with the problem.

**Road map.** Briefly tell the reader how the notebook moves from data and assumptions to mathematics, implementation, evaluation, failure analysis, and a decision.

## 1. Investigation contract

**STUDENT WORK.** Identify all of the following before running a model:

- the observation unit and population of interest;
- the target, structure, or quantity to be learned;
- information available at the moment of prediction or analysis;
- the consequences of a useful result and of a wrong result;
- whether this is exploratory, predictive, descriptive, or causal work; and
- why the selected algorithm or algorithm comparison can address the question.

State important non-goals. Predictive association alone does not establish causation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from dsm_project.algorithms import (
    create_student_algorithm,
    registered_student_algorithms,
)
from dsm_project.config import load_config
from dsm_project.contracts import TaskKind
from dsm_project.data import load_project_dataset
from dsm_project.experiments import (
    evaluate_supervised_estimator,
    fit_supervised_candidate,
)

# STUDENT WORK: point to the versioned project configuration. The example
# value exists only so the untouched template runs from a clean checkout.
CONFIG_PATH = Path("configs/example.toml")
TASK_KEY = "regression"
STUDENT_ALGORITHM = "batch_linear_regression"
TRUSTED_REFERENCE = "linear_regression"

config = load_config(CONFIG_PATH)
dataset = load_project_dataset(config)
task = dataset.tasks[TASK_KEY]
frame = dataset.observations

print("provenance:", dataset.provenance)
print("task:", task)
print("registered student algorithms:", registered_student_algorithms())

## 2. Data provenance and fitness for purpose

**STUDENT WORK.** Give the source URL, retrieval date, license or terms, sampling process, time and geographic scope, inclusion criteria, and known limitations. Explain how the package adapter validates the raw boundary. Link to the completed dataset card.

Do not list columns without interpretation. State what the variables measure, their units, when they become available, and which fields could leak future or target information. Explain why these observations can—or cannot—support the governing question.

In [ ]:
# A compact structural audit is more useful than printing the whole dataset.
audit = pd.DataFrame(
    {
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "unique": frame.nunique(dropna=False),
    }
)
print(f"rows={len(frame):,}, columns={frame.shape[1]}")
display(frame.head())
display(audit)

## 3. Purposeful exploration

**STUDENT WORK.** Replace the generic preview below with a small set of tables and plots that answer named questions about the proposed method. Useful questions concern scale, skew, outliers, missingness, class balance, dependence, time order, group structure, and possible leakage.

Before each display, say why it is needed. After it, interpret what changed in the modeling plan and what the evidence does not establish. Label variables, units, and categories. A gallery of plots without decisions is not analysis.

In [ ]:
selected = list(task.feature_columns[:4])
if task.target_column is not None:
    selected.append(task.target_column)

display(frame.loc[:, selected].describe(include="all").T)
frame.loc[:, selected].hist(figsize=(12, 7), bins=18, color="#1b9e77")
plt.suptitle("STUDENT WORK: replace with a question-driven figure")
plt.tight_layout();

## 4. Experimental boundaries and baseline

**STUDENT WORK.** Define which observations fit parameters, which guide hyperparameters and design choices, and which remain sealed for final evaluation. Justify random, stratified, grouped, temporal, spatial, or nested resampling from the data-generating process. State where preprocessing is fitted and how leakage is prevented.

Name a simple baseline that represents the minimum useful behavior. Define the primary metric mathematically, give its units or range, and explain why it matches real error costs. Add secondary metrics only when they answer different questions.

## 5. Mathematical model

**STUDENT WORK.** Teach the method before calling it. Define every symbol and dimension, then give the prediction or representation rule, objective function or update rule, assumptions, fitted state, hyperparameters, and computational cost. Derive the central update when it was derived in the course.

Connect the mathematics to this dataset: identify what $x_i$, $y_i$, parameters, distances, probabilities, margins, clusters, or representations mean here. Explain what changes when multiple algorithms are composed or compared. Cite the original method or an authoritative source.

## 6. Student implementation and verification

**STUDENT WORK.** Name and link the student-authored package class or function. Describe its public interface, fitted attributes, type and shape contracts, numerical safeguards, reproducibility controls, and expected exceptions. Summarize the unit, analytic, property, failure-mode, and differential tests that support it.

The next cell demonstrates the professional boundary: the notebook imports reusable code; it does not define the estimator. The trusted implementation is a measuring instrument, not a substitute for student work. Adapt the pipeline and parameters to the method rather than scaling automatically.

In [ ]:
if task.kind is TaskKind.UNSUPERVISED:
    raise ValueError(
        "Replace this supervised evaluation cell with the package's "
        "unsupervised experiment and stability evidence."
    )

# STUDENT WORK: justify each transformation and estimator parameter.
student_estimator = make_pipeline(
    StandardScaler(),
    create_student_algorithm(STUDENT_ALGORITHM),
)
student_result = evaluate_supervised_estimator(
    dataset,
    config,
    task_name=TASK_KEY,
    estimator_name=f"student:{STUDENT_ALGORITHM}",
    estimator=student_estimator,
)
reference_result = fit_supervised_candidate(
    dataset,
    config,
    task_name=TASK_KEY,
    model_name=TRUSTED_REFERENCE,
)

comparison = pd.DataFrame(
    [
        {"implementation": result.model_name, result.metric_name: result.metric_value}
        for result in (student_result, reference_result)
    ]
)
display(comparison)

## 7. Results, uncertainty, and practical meaning

**STUDENT WORK.** Report enough evidence to reproduce the comparison: data version, split or folds, seed policy, preprocessing, hyperparameters, sample counts, runtime or resource cost, and metrics. Quantify uncertainty across appropriate resamples or intervals. Preserve losing candidates.

Interpret effect size and practical meaning, not only rank. Explain whether disagreement with the trusted reference indicates a defect, an objective mismatch, or a legitimate design difference. Do not inspect the sealed final test set repeatedly while making choices.

In [ ]:
# This generic diagnostic is only a starting point. Choose diagnostics that
# expose the failures that matter for the declared user and task.
observed = student_result.observed
predicted = student_result.predictions

if task.kind is TaskKind.REGRESSION:
    residuals = observed - predicted
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(observed, predicted, alpha=0.7, color="#377eb8")
    limits = [min(observed.min(), predicted.min()), max(observed.max(), predicted.max())]
    axes[0].plot(limits, limits, "--", color="black")
    axes[0].set(xlabel="observed target", ylabel="predicted target", title="Observed versus predicted")
    axes[1].scatter(predicted, residuals, alpha=0.7, color="#e41a1c")
    axes[1].axhline(0.0, linestyle="--", color="black")
    axes[1].set(xlabel="predicted target", ylabel="residual", title="Residual structure")
    plt.tight_layout()
else:
    confusion = pd.crosstab(
        pd.Series(observed, name="observed"),
        pd.Series(predicted, name="predicted"),
        margins=True,
    )
    display(confusion)

## 8. Failure analysis

**STUDENT WORK.** Locate and characterize failures rather than merely acknowledging them. Examine relevant residual patterns, confusion-matrix cells, probability calibration, thresholds, subgroups, time periods, outliers, cluster instability, reconstruction errors, sensitivity to seeds, or compute limits. Relate failure modes to real consequences.

Distinguish problems caused by data coverage, measurement, task formulation, model assumptions, implementation defects, and distribution shift. State which failures monitoring could detect after deployment and which require delayed labels or scientific follow-up.

## 9. Conclusion and production decision

**STUDENT WORK.** Answer the governing question in bounded language. State the strongest supported claim, the most important limitation, and the next experiment most likely to change the conclusion. Decide whether the method should be rejected, retained as a baseline, investigated further, or promoted into the product.

If promoting it, identify the artifact, API, monitoring, fallback, and rollback consequences. If not, preserve the negative result and explain what it taught the team.

## Definition of done

Before requesting review, verify that:

- every **STUDENT WORK** prompt has been replaced;
- the notebook has one governing question and a readable narrative arc;
- notation, variables, units, assumptions, and metrics are defined;
- every important code cell is introduced and every result interpreted;
- student and trusted implementations are labeled and compared fairly;
- reusable behavior lives in `src/` or `scripts/` and is tested;
- figures are purposeful, labeled, readable, and discussed;
- uncertainty, failure cases, limitations, and responsible use are explicit;
- the conclusion answers the opening question without overstating evidence; and
- a clean **DSM Project** kernel runs every cell from top to bottom.